# Technology — System Design: Design a Data Platform
---

<div style="padding:15px;border-left:8px solid #a18cd1;background:#f5f0ff;border-radius:4px;">
<strong>Core Insight:</strong> This is Sean's actual Citi architecture. The design exists.
Kafka → Flink/Spark Streaming → S3 Parquet → Glue → Athena → Grafana.
Know every trade-off: Athena (serverless, pay-per-query) vs Redshift (provisioned, fast BI).
Own this design — you built it.
</div>

### The Architecture Overview
```
[APM Agents]
     │  (metrics, logs, traces)
     ▼
 [Kafka]          ← message bus, replay buffer, decoupling
     │
     ├──► [Flink Streaming]   ← real-time alerting (< 30s latency)
     │         │
     │         └──► [Alert DB / PagerDuty]
     │
     └──► [Spark Batch ETL]   ← hourly/daily aggregations
               │
               └──► [S3 Parquet] ← data lake (partitioned by date/env)
                         │
                         └──► [Glue Catalog] ← metadata layer
                                   │
                                   ├──► [Athena]    ← ad-hoc SQL queries
                                   └──► [Grafana]   ← dashboards
```

## 🏗️ Layer-by-Layer: What Each Component Does

| Layer | Component | Role | Why this choice |
|-------|-----------|------|----------------|
| **Ingestion** | Kafka | Message bus, replay buffer | Decouples producers from consumers; replay on failure |
| **Real-time** | Flink | Stream processing | Sub-second latency; stateful windows for anomaly detection |
| **Batch** | Spark | Large-scale ETL | Process 50GB/day efficiently; familiar to the team |
| **Storage** | S3 Parquet | Data lake | Cheap ($0.023/GB/month), columnar for analytics, durable |
| **Catalog** | AWS Glue | Metadata + schema | Athena and Spark read the same schema — one source of truth |
| **Query** | Athena | Ad-hoc SQL | Serverless, $5/TB scanned, no cluster to manage |
| **BI** | Redshift | Complex reports | Provisioned cluster, faster for repeated BI workloads |
| **Viz** | Grafana | Dashboards | Connects to Athena, Prometheus, Elasticsearch |

### The Two Query Paths
- **Ad-hoc / exploratory:** Athena → pay per query, no setup, slow for complex aggregations
- **Repeated / complex BI:** Redshift → faster, predictable cost, needs cluster management

## ⚖️ Key Design Trade-offs

### 1. Kafka vs Direct S3 Ingest
| Kafka | Direct S3 |
|-------|-----------|
| Replay capability — reprocess if consumer fails | No replay — data lost if consumer fails |
| Decouples producers (APM agents) from consumers (Spark) | Tight coupling |
| Adds operational complexity (Kafka cluster management) | Simple push to S3 |
| **Choose Kafka** when: consumers can fail and replay is critical | **Choose Direct S3** when: data is already durable and loss is acceptable |

**Citi decision:** Kafka, because APM agents can't buffer locally — if the Spark job fails at 2am,
Kafka holds the last 7 days of metrics and Spark replays from the last checkpoint.

### 2. Athena vs Redshift
| Athena | Redshift |
|--------|----------|
| Serverless — no cluster to manage | Provisioned — cluster cost even when idle |
| $5/TB scanned — expensive for repeated large queries | Fixed cost — predictable for heavy BI |
| Slow for complex aggregations across large data | Fast for complex BI queries (columnar + indexes) |
| **Choose Athena** for: ad-hoc exploration, infrequent queries | **Choose Redshift** for: repeated BI dashboards, complex joins |

**Citi decision:** Athena for ad-hoc capacity queries ($5/TB is fine for monthly reports).
Redshift would be overkill — only the capacity team and two analysts use it.

### 3. Lambda vs Kappa Architecture
| Lambda | Kappa |
|--------|-------|
| Two paths: batch + stream processed separately | Single path: stream only, reprocess by replaying |
| Complex: maintain two codebases | Simpler: one codebase |
| Batch is authoritative, stream is approximate | Stream is authoritative |
| **Citi uses Lambda:** batch Spark for historical accuracy, Flink for real-time alerting |

## 🎤 Interview Framework: "Design a Monitoring Data Platform"

### The 5-Step Answer Structure

**Step 1: Clarify requirements (2 minutes)**
- How many data sources? (6,000 servers → 50GB/day)
- Latency requirements? (real-time alerting < 30s, reports = next-day OK)
- Query patterns? (ad-hoc exploration vs repeated BI dashboards)
- Budget constraints? (serverless preferred for unpredictable workloads)
- Data retention? (90 days hot, 3 years cold → S3 lifecycle policy)

**Step 2: Identify the two query types**
- Real-time: anomaly detection, alerts (stream processing → Flink/Kafka Streams)
- Batch: trend analysis, capacity reports (batch → Spark → S3 → Athena)

**Step 3: Design the ingestion layer**
- APM agents → Kafka (replay buffer)
- Why Kafka: producers and consumers are decoupled; consumers can fail and replay

**Step 4: Design the storage layer**
- S3 Parquet, partitioned by `date=YYYY-MM-DD/env=prod/`
- Partition by date: monthly reports only scan 30 partitions of 365
- Glue Catalog: one schema definition, read by both Athena and Spark

**Step 5: Address scaling and failure modes**
- Kafka retention: 7 days — allows full replay of any week
- S3 lifecycle: 90-day hot (Parquet), then move to Glacier for cold storage
- Failure mode: if Spark job fails → restart from Kafka offset, not from S3
- Cost: Athena $5/TB × estimated monthly scan volume = projected monthly cost

## 📊 Data Partitioning Strategy

### Why Partitioning Matters
Without partitioning, every Athena query scans the entire dataset.
With date partitioning, a "last 7 days" query scans only 7 of 365 directories.

```
s3://citi-monitoring/
└── date=2024-01-15/
    └── env=prod/
        └── part-00000.parquet   (6,000 servers × 1,440 minutes = 8.6M rows/file)
└── date=2024-01-16/
    └── env=prod/
        └── part-00000.parquet
```

### The Partition Key Decision
| Partition by | Good for | Bad for |
|-------------|----------|---------|
| `date` only | Date range queries | Can't filter by env efficiently |
| `date` + `env` | Date + env filters | Too many small partitions if many envs |
| `env` + `date` | Env-first queries | Date queries scan all envs |

**Citi choice:** `date=YYYY-MM-DD/env={dev,staging,prod}/`
- Monthly reports filter by date → partition pruning works
- Env-specific debugging filters by env → second partition helps
- 3 environments = manageable partition count

### File Size Rule
Target: 128MB-512MB per Parquet file. Too small: too many S3 API calls. Too large: can't parallelize.
Command: `spark.coalesce(4)` before writing → 4 files per partition.

## 🎤 5 Interview Q&A

**Q1: Why use Kafka instead of writing directly to S3?**
Kafka provides a replay buffer. If the Spark job fails at 2am, it can restart and read
from its last committed Kafka offset — reprocessing only the missed data.
With direct S3 writes, if the APM agent crashes mid-write, data is lost.
Kafka also decouples producers from consumers: APM agents don't need to know about Spark;
they just publish to a topic. New consumers (Flink for real-time alerts) can be added
without changing the agents.

---

**Q2: What is the difference between Athena and Redshift and when do you choose each?**
Athena: serverless, queries S3 directly, pay per TB scanned ($5/TB). No cluster to manage.
Best for: ad-hoc exploration, infrequent queries, teams that can't afford idle cluster cost.
Redshift: provisioned cluster, much faster for complex multi-table BI queries, fixed cost.
Best for: daily BI dashboards with complex aggregations, teams running 100+ queries/day.
Rule: if query frequency × cost per query > cluster monthly cost → use Redshift.

---

**Q3: What is the Lambda architecture and what are its trade-offs?**
Lambda has two processing paths: batch (slow, accurate, historical) and stream (fast, approximate, real-time).
Results from both are merged in a serving layer. Trade-off: you maintain two codebases for the same logic.
The Kappa architecture simplifies this by using only streaming — historical reprocessing = replaying Kafka.
Citi uses Lambda because: (1) batch Spark produces authoritative numbers for regulatory reporting,
(2) Flink provides real-time alerts that can be slightly approximate, (3) team had both Spark and
stream processing expertise.

---

**Q4: How do you handle schema evolution in a data lake?**
Schema evolution means adding/removing/changing columns over time.
Parquet supports column addition: new files can have extra columns; old files return NULL for new columns.
Glue Catalog tracks schema versions — Athena reads the current schema and handles nulls for old files.
Dangerous changes: renaming or removing columns breaks backward compatibility.
Strategy: (1) only ADD columns (never remove), (2) version your schema: `v1/`, `v2/`, (3) use
Glue Schema Registry for enforcing Avro/Protobuf schemas at ingestion time.

---

**Q5: How do you optimize Athena query costs?**
Athena charges per TB scanned. Reduce cost with:
(1) Parquet format — columnar, only reads requested columns (vs CSV which reads all columns)
(2) Partition pruning — `WHERE date = '2024-01-15'` reads only that day's partition
(3) Compression — Parquet + Snappy reduces file size by ~75% vs raw CSV
(4) Partition projection — register partitions in Glue to avoid full metadata scans
(5) SELECT only needed columns — columnar format skips unselected columns entirely
Rule of thumb: same query on Parquet vs CSV = 10-20x cost reduction.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Data Lake** | S3 storage of raw + processed data in open formats (Parquet, ORC) — cheap, scalable |
| **Data Warehouse** | Optimized for SQL analytics — Redshift, Snowflake, BigQuery. Faster for BI, higher cost |
| **Lambda Architecture** | Two-path processing: batch (accurate, slow) + stream (fast, approximate) |
| **Kappa Architecture** | Single stream path — replay Kafka for reprocessing. Simpler, requires fast stream |
| **Kafka Offset** | Position in a Kafka topic partition — consumers commit their offset to resume after failure |
| **Partition Pruning** | Athena/Spark skips S3 directories that don't match the WHERE clause partition key |
| **Parquet** | Columnar storage format — reads only needed columns, high compression, fast for analytics |
| **AWS Glue** | Managed ETL service + data catalog (schema registry for S3 data) |
| **Athena** | Serverless SQL on S3 — pay per TB scanned ($5/TB), no cluster management |
| **Redshift** | AWS data warehouse — fast for complex BI, provisioned cluster |
| **Flink** | Stream processing framework — low-latency stateful computation, exactly-once semantics |
| **Schema Evolution** | Adding/changing columns over time — Parquet + Glue handle column additions gracefully |

## 💼 The Citi Narrative

**Context:** Citi monitoring data was siloed — CA APM for some apps, AppDynamics for others,
no unified platform for cross-application capacity analysis.

**The Problem:** Generating a monthly capacity report required manually extracting data from
4 APM tools, joining in Excel, and building charts. This took 2-3 analysts a full day.

**The Architecture Designed:**
- Kafka: unified ingestion bus — each APM tool's agent published to a topic
- Spark Streaming: hourly ETL → S3 Parquet, partitioned by date/env
- Glue Catalog: single schema across all tables
- Athena: ad-hoc capacity queries (SQL, no cluster to maintain)
- Grafana: dashboards reading from Athena via the Athena-Grafana plugin

**Key Trade-off Made:** Chose Athena over Redshift. Volume was ~50GB/day → monthly query
cost = ~$15 (3TB/month × $5/TB). Redshift cluster would cost $180+/month. For 2-3 analysts
running monthly reports, Athena was 10x cheaper.

**Result:**
- Monthly capacity report: 2 hours (from 2 analyst-days)
- Ad-hoc queries: available to anyone with Athena access, no request queue
- New APM tool onboarding: add a Kafka producer → zero changes to the query/dashboard layer

**Interview Line:** *"We replaced 4 separate APM reporting processes with one platform.
The key architectural decision was Kafka for ingestion — it meant we could add or remove
APM tools without touching the downstream pipeline. The capacity report went from 2 days to 2 hours."*

## 🎯 Summary

### The Architecture (Memorize This)
```
APM Agents → Kafka → Spark/Flink → S3 Parquet → Glue → Athena/Redshift → Grafana
```

### The Five Trade-offs (Always Discuss These)
1. **Kafka vs Direct S3:** Replay capability vs simplicity — choose Kafka when consumers fail
2. **Athena vs Redshift:** Serverless pay-per-query vs provisioned fast BI — use Athena for ad-hoc
3. **Lambda vs Kappa:** Two codebases vs one — Lambda when batch accuracy matters for regulatory data
4. **Date partition vs no partition:** 10-50x cost reduction on Athena — always partition by date
5. **Parquet vs CSV:** 10-20x Athena cost reduction — always use Parquet + Snappy

### The Numbers (Know These)
- S3: $0.023/GB/month
- Athena: $5/TB scanned
- Parquet compression: ~75% vs CSV
- Citi monthly query cost: ~$15 (vs $180+/month for Redshift)
- Report time: 2 hours (down from 2 analyst-days)

### Interview Confidence Checklist
- [ ] Can draw the full architecture from memory (Kafka → S3 → Athena)
- [ ] Can explain Athena vs Redshift trade-off with real numbers
- [ ] Can explain why Kafka instead of direct S3
- [ ] Can explain partition pruning and why it matters for cost
- [ ] Can name the Citi narrative: 4 APM tools → 1 platform, report time 2 days → 2 hours

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀